NTSB Aviation Accident Database — Data Loading

This notebook converts the raw Microsoft Access database (`avall.mdb`)
distributed by the NTSB into a local SQLite file used by the analysis notebook.

**Run this once.** It does not need to be re-run unless the source data changes.

**Source:** [NTSB Aviation Accident Database](https://www.ntsb.gov/safety/data/Pages/Data_Stats.aspx)
**File:** `avall.zip` → extract `avall.mdb` → place in `../data/`

In [6]:
import subprocess
import sqlite3
import pandas as pd
from io import StringIO
import os

# Create required directories if they don't exist
for folder in ["../data", "../charts"]:
    os.makedirs(folder, exist_ok=True)
    print(f"✔  {folder}")

MDB = "../data/avall.mdb"
DB  = "../data/aviation.db"

✔  ../data
✔  ../charts


In [7]:
# Extract table names from the Access file
result = subprocess.run(["mdb-tables", "-1", MDB], capture_output=True, text=True)
table_names = [t for t in result.stdout.strip().split("\n") if t]
print(f"Tables found: {len(table_names)}")

Tables found: 20


In [8]:
# Export each table to SQLite
conn = sqlite3.connect(DB)

for table in table_names:
    result = subprocess.run(["mdb-export", MDB, table], capture_output=True, text=True)
    if not result.stdout.strip():
        print(f"⚠  {table:<30} empty — skipped")
        continue
    df = pd.read_csv(StringIO(result.stdout), low_memory=False)
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(f"✔  {table:<30} {len(df):>7,} rows  |  {len(df.columns)} columns")

conn.close()
print("\nDone. aviation.db is ready.")

✔  Country                            262 rows  |  2 columns
✔  ct_iaids                           955 rows  |  11 columns
✔  ct_seqevt                        2,224 rows  |  2 columns
✔  dt_events                      112,967 rows  |  5 columns
✔  dt_Flight_Crew                 173,638 rows  |  7 columns
✔  eADMSPUB_DataDictionary          4,574 rows  |  13 columns
✔  engines                         27,723 rows  |  17 columns
✔  events                          30,212 rows  |  73 columns
✔  Events_Sequence                 64,642 rows  |  10 columns
✔  Flight_Crew                     31,545 rows  |  33 columns
✔  flight_time                    396,507 rows  |  8 columns
✔  injury                         173,347 rows  |  7 columns
✔  NTSB_Admin                      30,212 rows  |  5 columns
✔  Occurrences                          0 rows  |  8 columns
✔  seq_of_events                        0 rows  |  11 columns
✔  states                              51 rows  |  3 columns
✔  aircraft      